In [0]:
%run ../source_to_bronze/utils

In [0]:
silver_path = "/Volumes/main/default/assignment_data/silver/Employee_info/dim_employee"

employee_gold_df = (
    spark.read
    .format("delta")
    .load(silver_path)
)

display(employee_gold_df)

In [0]:
from pyspark.sql.functions import (
    sum,
    count,
    avg,
    col,
    current_date
)

In [0]:
department_salary_df = (
    employee_gold_df
    .groupBy("department")
    .agg(
        sum("salary").alias("total_salary")
    )
    .orderBy(
        col("total_salary").desc()
    )
)

display(department_salary_df)

In [0]:
employee_count_df = (
    employee_gold_df
    .groupBy(
        "department",
        "country"
    )
    .agg(
        count("employee_i_d").alias("employee_count")
    )
)

display(employee_count_df)

In [0]:
department_country_df = (
    employee_gold_df
    .select(
        "department",
        "country"
    )
    .distinct()
)

display(department_country_df)

In [0]:
average_age_df = (
    employee_gold_df
    .groupBy("department")
    .agg(
        avg("age").alias("average_age")
    )
)

display(average_age_df)

In [0]:
gold_df = employee_gold_df.withColumn(
    "at_load_date",
    current_date()
)

display(gold_df)

In [0]:
from datetime import date

gold_path = "/Volumes/main/default/assignment_data/gold/employee/fact_employee"

load_date = date.today().isoformat()

(
    gold_df.write
    .format("delta")
    .mode("overwrite")
    .option(
        "replaceWhere",
        f"at_load_date = '{load_date}'"
    )
    .save(gold_path)
)

In [0]:
gold_check_df = (
    spark.read
    .format("delta")
    .load(gold_path)
)

display(gold_check_df)